In [ ]:
!pip install -U "huggingface_hub" h5py
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
!pip3 install diffusers scikit-image scikit-video numcodecs
!pip install transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 7.7 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.16.1
    Uninstalling huggingface_hub-1.16.1:
      Successfully uninstalled huggingface_hub-1.16.1
Looking in indexes: https://download.pytorch.org/whl/cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 122.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 104.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 43.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.16.4
    Uninstalling huggingface_hub-1.16.4:
      Successfully uninstalled huggingface_hub-1.16.4
  

In [ ]:
!git clone https://github.com/GoncaloMark/DuPLO-VLA.git

Cloning into 'DuPLO-VLA'...
remote: Enumerating objects: 1555, done.
remote: Counting objects: 100% (270/270), done.
remote: Compressing objects: 100% (210/210), done.
remote: Total 1555 (delta 131), reused 146 (delta 60), pack-reused 1285 (from 1)
Receiving objects: 100% (1555/1555), 166.08 MiB | 33.66 MiB/s, done.
Resolving deltas: 100% (719/719), done.


In [ ]:
%cd DuPLO-VLA
!git switch v2

/content/DuPLO-VLA
Already on 'v2'
Your branch is up to date with 'origin/v2'.


In [ ]:
!git pull

Already up to date.


In [ ]:
from google.colab import userdata
from huggingface_hub import login, hf_hub_download
import sys, os, copy

from typing import Tuple, Sequence, Dict, Union, Optional, Callable
import numpy as np
import math
import torch
import torch.nn as nn
import torchvision
import collections
from diffusers.schedulers.scheduling_ddpm import DDPMScheduler
from diffusers.training_utils import EMAModel
from diffusers.optimization import get_scheduler
from tqdm.auto import tqdm

import matplotlib.pyplot as plt

import gym
from gym import spaces
import cv2
import skimage.transform as st
from skvideo.io import vwrite
from IPython.display import Video

import h5py

import shutil
import math

sys.path.append('/content/DuPLO-VLA/src')
os.chdir('/content/DuPLO-VLA')

import torch.nn.functional as F

from torch.utils.tensorboard import SummaryWriter

from vlm.latent_encoder import LatentTaskEncoder
from action_heads.diffusion import RDTPolicy
from vlm.aux_action import ActionPredictorAuxHead


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/skvideo/utils/stpyr.py:2: DeprecationWarning: scipy.misc is deprecated and will be removed in 2.0.0
  import scipy.misc as sc
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future ver

In [ ]:
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Successfully logged in to Hugging Face!")
except userdata.SecretNotFoundError:
    print("HF_TOKEN not found in Colab secrets. Please add it for a stable connection.")

Successfully logged in to Hugging Face!


In [ ]:
repo_id = "lnxdre4d/libero_features_v2"
filename = "libero_features_v2.h5"

In [ ]:
local_file = hf_hub_download(repo_id=repo_id, filename=filename, local_dir="/mnt/local-scratch/", repo_type="dataset")
print(f"File downloaded to: {local_file}")


libero_features_v3.h5:   0%|          | 0.00/115G [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1727: DeprecationWarning: hf_xet.download_files() is deprecated. Use XetSession().new_file_download_group().start_download_file() instead.
  xet_get(


File downloaded to: /mnt/local-scratch/libero_features_v3.h5


In [ ]:
def create_sample_indices(
        episode_ends:np.ndarray, sequence_length:int,
        pad_before: int=0, pad_after: int=0):
    indices = list()
    for i in range(len(episode_ends)):
        start_idx = 0
        if i > 0:
            start_idx = episode_ends[i-1]
        end_idx = episode_ends[i]
        episode_length = end_idx - start_idx

        min_start = -pad_before
        max_start = episode_length - sequence_length + pad_after

        # range stops one idx before end
        for idx in range(min_start, max_start+1):
            buffer_start_idx = max(idx, 0) + start_idx
            buffer_end_idx = min(idx+sequence_length, episode_length) + start_idx
            start_offset = buffer_start_idx - (idx+start_idx)
            end_offset = (idx+sequence_length+start_idx) - buffer_end_idx
            sample_start_idx = 0 + start_offset
            sample_end_idx = sequence_length - end_offset
            indices.append([
                buffer_start_idx, buffer_end_idx,
                sample_start_idx, sample_end_idx])
    indices = np.array(indices)
    return indices


def sample_sequence(train_data, sequence_length,
                    buffer_start_idx, buffer_end_idx,
                    sample_start_idx, sample_end_idx):
    result = dict()
    for key, input_arr in train_data.items():
        sample = input_arr[buffer_start_idx:buffer_end_idx]
        data = sample
        if (sample_start_idx > 0) or (sample_end_idx < sequence_length):
            data = np.zeros(
                shape=(sequence_length,) + input_arr.shape[1:],
                dtype=input_arr.dtype)
            if sample_start_idx > 0:
                data[:sample_start_idx] = sample[0]
            if sample_end_idx < sequence_length:
                data[sample_end_idx:] = sample[-1]
            data[sample_start_idx:sample_end_idx] = sample
        result[key] = data
    return result

# normalize data
def get_data_stats(data):
    data = data.reshape(-1,data.shape[-1])
    stats = {
        'min': np.min(data, axis=0),
        'max': np.max(data, axis=0)
    }
    return stats

def normalize_data(data, stats):
    # nomalize to [0,1]
    ndata = (data - stats['min']) / (stats['max'] - stats['min'])
    # normalize to [-1, 1]
    ndata = ndata * 2 - 1
    return ndata

def unnormalize_data(ndata, stats):
    ndata = (ndata + 1) / 2
    data = ndata * (stats['max'] - stats['min']) + stats['min']
    return data

In [ ]:
class LiberoLatentDataset(torch.utils.data.Dataset):
    def __init__(
        self,
        dataset_path: str,
        pred_horizon: int,
        obs_horizon: int,
        proprio_horizon: int,
        action_horizon: int,
        text_embeddings_path: str = None,
    ):
        self.dataset_path = dataset_path

        with h5py.File(dataset_path, "r") as f:
            dataset_root = f["data"]
            self.demo_keys = sorted(
                [k for k in dataset_root.keys() if k.startswith("demo_")],
                key=lambda x: int(x.split("_")[1]),
            )

            first = dataset_root[self.demo_keys[0]]
            self._latent_item_shape = first["obs/img_features"].shape[1:]
            self._mask_item_shape   = first["text_mask"].shape[1:]
            latent_dtype = first["obs/img_features"].dtype
            mask_dtype   = first["text_mask"].dtype
            state_dtype  = first["obs/state"].dtype
            action_dtype = first["action"].dtype
            state_dim    = first["obs/state"].shape[1]
            action_dim   = first["action"].shape[1]

            total_frames = 0
            total_slow   = 0
            per_demo_lengths = []
            for key in tqdm(self.demo_keys, desc="Scanning sizes"):
                grp = dataset_root[key]
                T      = grp["action"].shape[0]
                n_slow = grp["obs/img_features"].shape[0]
                per_demo_lengths.append((T, n_slow))
                total_frames += T
                total_slow   += n_slow

            print(f"Total frames: {total_frames}, total slow: {total_slow}")
            print(f"Allocating latents: ({total_slow},) + {self._latent_item_shape} {latent_dtype}")

            self._latents_np = np.empty(
                (total_slow,) + self._latent_item_shape, dtype=latent_dtype
            )
            self._masks_np = np.empty(
                (total_slow,) + self._mask_item_shape, dtype=mask_dtype
            )

            agent_pos_np = np.empty((total_frames, state_dim), dtype=state_dtype)
            action_np    = np.empty((total_frames, action_dim), dtype=action_dtype)

            self.local_vlm_frame_idx = np.empty(total_frames, dtype=np.int64)
            self.demo_id_per_frame   = np.empty(total_frames, dtype=np.int64)

            per_demo_slow_offsets = np.empty(len(self.demo_keys), dtype=np.int64)
            per_demo_frame_starts = np.empty(len(self.demo_keys), dtype=np.int64)
            episode_ends = np.empty(len(self.demo_keys), dtype=np.int64)

            frame_offset = 0
            slow_offset  = 0
            for demo_id, key in enumerate(
                tqdm(self.demo_keys, desc="Loading h5 \u2192 preallocated arrays")
            ):
                grp = dataset_root[key]
                T, n_slow = per_demo_lengths[demo_id]

                self._latents_np[slow_offset : slow_offset + n_slow] = grp["obs/img_features"][:]
                self._masks_np[slow_offset : slow_offset + n_slow]   = grp["text_mask"][:]
                agent_pos_np[frame_offset : frame_offset + T]        = grp["obs/state"][:]
                action_np[frame_offset : frame_offset + T]           = grp["action"][:]

                self.local_vlm_frame_idx[frame_offset : frame_offset + T] = \
                    grp["vlm_frame_idx"][:].astype(np.int64)
                self.demo_id_per_frame[frame_offset : frame_offset + T] = demo_id

                per_demo_slow_offsets[demo_id] = slow_offset
                per_demo_frame_starts[demo_id] = frame_offset
                episode_ends[demo_id]          = frame_offset + T

                frame_offset += T
                slow_offset  += n_slow

        self.per_demo_slow_offsets = per_demo_slow_offsets
        self.per_demo_frame_starts = per_demo_frame_starts
        self.episode_ends = episode_ends

        print("Converting latents to shared-memory tensor...")
        self.all_latents = torch.from_numpy(self._latents_np.view(np.int16))
        del self._latents_np

        print("Converting masks to shared-memory tensor...")
        self.all_text_masks = torch.from_numpy(self._masks_np)
        del self._masks_np

        latent_gb = self.all_latents.numel() * self.all_latents.element_size() / 1e9
        mask_mb   = self.all_text_masks.numel() * self.all_text_masks.element_size() / 1e6
        print(f"Shared latents: {tuple(self.all_latents.shape)} ({latent_gb:.1f} GB)")
        print(f"Shared masks:   {tuple(self.all_text_masks.shape)} ({mask_mb:.1f} MB)")

        # Windows need to be large enough to cover BOTH obs_horizon and
        # proprio_horizon at the start, plus pred_horizon's action span.
        # pad_before is sized to max(obs, proprio) - 1
        # pad_after sized for the action span. Then we filter out any window
        # that requires padding at either end.
        # Result: every kept window has full obs/proprio
        # history and a full action chunk.
        pad_before = max(obs_horizon, proprio_horizon) - 1
        pad_after  = action_horizon - 1

        raw_indices = create_sample_indices(
            episode_ends     = self.episode_ends,
            sequence_length  = pred_horizon,
            pad_before       = pad_before,
            pad_after        = pad_after,
        )

        clean = []
        for ent in raw_indices:
            bs, be, ss, se = ent
            if ss == 0 and se == pred_horizon and be > bs:
                clean.append(ent)
        n_before = len(raw_indices)
        n_after  = len(clean)
        print(f"Window filter: {n_before} -> {n_after} "
              f"(dropped {n_before - n_after} that needed padding)")
        self.indices = clean

        train_data = {"agent_pos": agent_pos_np, "action": action_np}
        self.stats = {}
        self.normalized_train_data = {}
        for k in ("agent_pos", "action"):
            self.stats[k] = get_data_stats(train_data[k])
            self.normalized_train_data[k] = normalize_data(train_data[k], self.stats[k])

        self.pred_horizon    = pred_horizon
        self.obs_horizon     = obs_horizon
        self.proprio_horizon = proprio_horizon
        self.action_horizon  = action_horizon

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
      bs, be, ss, se = self.indices[idx]

      demo_id  = int(self.demo_id_per_frame[bs])
      demo_off = int(self.per_demo_slow_offsets[demo_id])

      vlm_local = self.local_vlm_frame_idx[bs : bs + self.obs_horizon]
      global_idx = torch.from_numpy(demo_off + vlm_local)

      latent_int16 = self.all_latents[global_idx]
      if not latent_int16.is_contiguous():
          latent_int16 = latent_int16.contiguous()
      latent_tensor = latent_int16.view(torch.bfloat16)

      mask_tensor = self.all_text_masks[global_idx].bool()

      proprio_slice = self.normalized_train_data["agent_pos"][
          bs : bs + self.proprio_horizon
      ]
      action_slice = self.normalized_train_data["action"][bs : be]

      return {
          "image_latent": latent_tensor,
          "text_mask":    mask_tensor,
          "agent_pos":    torch.from_numpy(proprio_slice).float(),
          "action":       torch.from_numpy(action_slice).float(),
      }


In [ ]:
src_path = "/mnt/local-scratch/libero_features_v2.h5"
dst_path = "/mnt/local-scratch/libero_features_uncompressed.h5"


def copy_attrs(src_obj, dst_obj):
    for k, v in src_obj.attrs.items():
        dst_obj.attrs[k] = v


def copy_dataset(src_ds, dst_grp, name):
    new_ds = dst_grp.create_dataset(
        name,
        data=src_ds[:],
        chunks=src_ds.chunks if src_ds.chunks else None,
        dtype=src_ds.dtype,
    )
    copy_attrs(src_ds, new_ds)
    return new_ds


with h5py.File(src_path, "r") as src, h5py.File(dst_path, "w") as dst:
    copy_attrs(src, dst)

    src_data = src["data"]
    dst_data = dst.create_group("data")
    copy_attrs(src_data, dst_data)

    for demo_key in tqdm(src_data.keys(), desc="Uncompressing"):
        src_grp = src_data[demo_key]
        dst_grp = dst_data.create_group(demo_key)
        copy_attrs(src_grp, dst_grp)

        for sub_key, sub_obj in src_grp.items():
            if isinstance(sub_obj, h5py.Group):
                dst_sub_grp = dst_grp.create_group(sub_key)
                copy_attrs(sub_obj, dst_sub_grp)
                for ds_name, ds in sub_obj.items():
                    copy_dataset(ds, dst_sub_grp, ds_name)
            else:
                copy_dataset(sub_obj, dst_grp, sub_key)


print("\nVerifying...")
with h5py.File(src_path, "r") as src, h5py.File(dst_path, "r") as dst:
    src_demos = sorted(src["data"].keys())
    dst_demos = sorted(dst["data"].keys())
    assert len(src_demos) == len(dst_demos), \
        f"demo count differs: {len(src_demos)} vs {len(dst_demos)}"

    demo0 = src_demos[0]
    src_attrs = dict(src["data"][demo0].attrs)
    dst_attrs = dict(dst["data"][demo0].attrs)
    assert src_attrs.keys() == dst_attrs.keys(), \
        f"attr keys differ: {src_attrs.keys()} vs {dst_attrs.keys()}"
    print(f"  Demo attrs preserved: {list(dst_attrs.keys())}")

    if "task_index" in dst_attrs:
        print(f"  Sample task_index: {dst_attrs['task_index']}")

    # Verify dtype on big dataset
    src_feats = src["data"][demo0]["obs/img_features"]
    dst_feats = dst["data"][demo0]["obs/img_features"]
    assert src_feats.shape == dst_feats.shape
    assert src_feats.dtype == dst_feats.dtype
    print(f"  img_features dtype preserved: {dst_feats.dtype}, shape: {dst_feats.shape}")

    print(f"  Total demos: {len(dst_demos)}")
    print("Verification passed.")

Uncompressing:   0%|          | 0/428 [00:00<?, ?it/s]


Verifying...
  Demo attrs preserved: ['base_instruction', 'instruction', 'num_frames', 'num_slow_frames', 'task_index', 'task_name']
  Sample task_index: 10
  img_features dtype preserved: uint16, shape: (14, 4, 846, 2560)
  Total demos: 428
Verification passed.


In [ ]:
device = torch.device("cuda")

obs_horizon      = 1
proprio_horizon  = 4
pred_horizon     = 16
action_horizon   = 8
action_dim       = 7
proprio_dim      = 8
state_dim        = proprio_horizon * proprio_dim

vlm_num_layers   = 4
vlm_hidden_dim   = 2560

num_queries      = 64
q_hidden_dim     = 512
latent_dim       = 512
encoder_dropout  = 0.1

num_train_timesteps     = 1000
num_inference_timesteps = 20
prediction_type         = "sample"
beta_schedule           = "squaredcos_cap_v2"

num_epochs       = 120
lr               = 2e-4
encoder_lr_mult  = 0.3
weight_decay     = 1e-6
grad_clip_norm   = 1.0
warmup_steps     = 500
batch_size       = 256

AUX_WEIGHT       = 0.5


In [ ]:
latent_encoder = LatentTaskEncoder(
    vlm_hidden_dim      = vlm_hidden_dim,
    num_layers          = vlm_num_layers,
    q_hidden_dim        = q_hidden_dim,
    latent_dim          = latent_dim,
    num_pooling_queries = num_queries,
    num_attention_heads = 8,
    dropout             = encoder_dropout,
).to(device)


In [ ]:
# No cond_seq_lens entry for "latent" as Q-Pooler queries are order-invariant.
policy = RDTPolicy(
    action_dim                = action_dim,
    horizon                   = pred_horizon,
    state_dim                 = state_dim,
    cond_dims                 = {"latent": latent_dim},
    cond_seq_lens             = {},
    hidden_dim                = 512,
    depth                     = 6,
    num_heads                 = 8,
    num_train_timesteps       = num_train_timesteps,
    num_inference_timesteps   = num_inference_timesteps,
    prediction_type           = prediction_type,
    beta_schedule             = beta_schedule,
).to(device)

In [ ]:
aux_head = ActionPredictorAuxHead(
    latent_dim = latent_dim,
    action_dim = action_dim,
    horizon    = pred_horizon,
    hidden_dim = 512,
    dropout    = 0.1,
).to(device)


In [ ]:
for var_name in ['ds', 'src_feats', 'dst_feats', 'src_grp', 'dst_grp', 'sub_obj', 'first']:
    if var_name in globals():
        del globals()[var_name]

dataset_path = "/mnt/local-scratch/libero_features_uncompressed.h5"

dataset = LiberoLatentDataset(
    dataset_path=dataset_path,
    pred_horizon=pred_horizon,
    obs_horizon=obs_horizon,
    action_horizon=action_horizon,
    proprio_horizon=proprio_horizon,
)

stats = dataset.stats


Scanning sizes:   0%|          | 0/428 [00:00<?, ?it/s]

Total frames: 52042, total slow: 6687
Allocating latents: (6687,) + (4, 846, 2560) uint16


Loading h5 → preallocated arrays:   0%|          | 0/428 [00:00<?, ?it/s]

Converting latents to shared-memory tensor...
Converting masks to shared-memory tensor...
Shared latents: (6687, 4, 846, 2560) (115.9 GB)
Shared masks:   (6687, 846) (5.7 MB)
Window filter: 49902 -> 45622 (dropped 4280 that needed padding)


In [ ]:
dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size         = 256,
    num_workers        = 8,
    prefetch_factor    = 4,
    shuffle            = True,
    pin_memory         = True,
    persistent_workers = True,
    drop_last          = True,
)


In [ ]:
param_groups = [
    {"params": latent_encoder.parameters(), "lr": lr * encoder_lr_mult},
    {"params": policy.parameters(), "lr": lr},
    {"params": aux_head.parameters(), "lr": lr},
]
optimizer = torch.optim.AdamW(param_groups, weight_decay=weight_decay)

lr_scheduler = get_scheduler(
    name="cosine",
    optimizer=optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=len(dataloader) * num_epochs,
)

ema_encoder = EMAModel(parameters=list(latent_encoder.parameters()), power=0.75)
ema_policy = EMAModel(parameters=list(policy.parameters()), power=0.75)
ema_aux = EMAModel(parameters=list(aux_head.parameters()), power=0.75)


In [ ]:
@torch.no_grad()
def gate_magnitudes(policy):
    t_test = torch.zeros(1, device=device, dtype=torch.long)
    t_emb = policy.t_embedder(t_test)
    gate_ca_means, gate_ca_maxes = [], []
    for block in policy.blocks:
        chunks = block.adaLN(t_emb).chunk(9, dim=-1)
        gate_ca = chunks[5]
        gate_ca_means.append(gate_ca.abs().mean().item())
        gate_ca_maxes.append(gate_ca.abs().max().item())
    return gate_ca_means, gate_ca_maxes



In [ ]:
!mkdir -p /content/runs/duplo_vla_libero_v2
%load_ext tensorboard
%tensorboard --logdir /content/runs/duplo_vla_libero_v2

Reusing TensorBoard on port 6006 (pid 7368), started 0:23:46 ago. (Use '!kill 7368' to kill it.)

<IPython.core.display.Javascript object>

In [ ]:
def ema_to_state_dict(ema_model, module):
    ema_model.store(module.parameters())
    ema_model.copy_to(module.parameters())
    out = {k: v.detach().clone() for k, v in module.state_dict().items()}
    ema_model.restore(module.parameters())
    return out

In [ ]:
writer = SummaryWriter(log_dir="/content/runs/duplo_vla_libero_v2")

global_step = 0

policy.train()
latent_encoder.train()
aux_head.train()

with tqdm(range(num_epochs), desc="Epochs") as t_global:
    for epoch_idx in t_global:
        for nbatch in dataloader:
            global_step += 1

            image_latent = nbatch["image_latent"].to(device, non_blocking=True)
            text_mask    = nbatch["text_mask"].to(device, non_blocking=True)
            nagent_pos   = nbatch["agent_pos"].to(device, non_blocking=True)
            naction      = nbatch["action"].to(device, non_blocking=True)

            B = image_latent.shape[0]

            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                enc_out = latent_encoder(
                    vlm_hidden_states=image_latent,
                    key_padding_mask=text_mask,
                )
                latent_seq = enc_out["latent_seq"]

                state = nagent_pos.reshape(B, -1)
                cond_dict = {"latent": latent_seq}
                policy_loss = policy(naction, state, cond_dict)

                aux_pred = aux_head(latent_seq)
                aux_loss = F.mse_loss(aux_pred, naction.to(aux_pred.dtype))

                total_loss = policy_loss + AUX_WEIGHT * aux_loss

            optimizer.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(
                list(latent_encoder.parameters())
                + list(policy.parameters())
                + list(aux_head.parameters()),
                grad_clip_norm,
            )
            optimizer.step()
            lr_scheduler.step()

            ema_encoder.step(latent_encoder.parameters())
            ema_policy.step(policy.parameters())
            ema_aux.step(aux_head.parameters())

            if global_step % 20 == 0:
                writer.add_scalar("loss/policy", policy_loss.item(), global_step)
                writer.add_scalar("loss/aux",    aux_loss.item(),    global_step)
                writer.add_scalar("loss/total",  total_loss.item(),  global_step)
                with torch.no_grad():
                    lstd = latent_seq.detach().float().std(dim=(0, 1)).mean().item()
                    writer.add_scalar("latent/per_dim_std_mean", lstd, global_step)

            if global_step % 200 == 0:
                means, maxes = gate_magnitudes(policy)
                for i, (m, mx) in enumerate(zip(means, maxes)):
                    writer.add_scalar(f"gate_ca_mean/block_{i}", m,  global_step)
                    writer.add_scalar(f"gate_ca_max/block_{i}",  mx, global_step)

            if global_step % 50 == 0:
                print(f"step {global_step} | policy={policy_loss.item():.4f} "
                      f"aux={aux_loss.item():.4f} total={total_loss.item():.4f}")

        t_global.set_postfix({
            "step":   global_step,
            "policy": f"{policy_loss.item():.4f}",
            "aux":    f"{aux_loss.item():.4f}",
        })

        if (epoch_idx + 1) % 10 == 0:
            ckpt = {
                "epoch":          epoch_idx + 1,
                "global_step":    global_step,
                "latent_encoder": latent_encoder.state_dict(),
                "policy":         policy.state_dict(),
                "aux_head":       aux_head.state_dict(),
                "ema_encoder":    ema_to_state_dict(ema_encoder, latent_encoder),
                "ema_policy":     ema_to_state_dict(ema_policy,  policy),
                "ema_aux":        ema_to_state_dict(ema_aux,     aux_head),
                "stats":          stats,
            }
            path = f"/content/DuPLO-VLA/duplo_libero_v2_epoch_{epoch_idx+1}.pt"
            torch.save(ckpt, path)
            print(f"Saved {path}")

writer.close()
print("Done.")